# 03 — Model Comparison

Train and compare four models for fraud detection: Logistic Regression (interpretable baseline), XGBoost (primary model), LightGBM (speed comparison), and Isolation Forest (unsupervised anomaly baseline). Evaluated primarily on Precision-Recall AUC, since accuracy is meaningless on this imbalanced dataset (see EDA/imbalance notebooks).

## 1. Load the Saved Splits

Load the train/calibration/test splits saved in `02_imbalance_strategies.ipynb`, so this notebook builds on the exact same data without repeating preprocessing.

In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_parquet('../data/processed/train.parquet')
calib_df = pd.read_parquet('../data/processed/calibration.parquet')
test_df = pd.read_parquet('../data/processed/test.parquet')

X_train, y_train = train_df.drop(columns=['Class']), train_df['Class']
X_calib, y_calib = calib_df.drop(columns=['Class']), calib_df['Class']
X_test, y_test = test_df.drop(columns=['Class']), test_df['Class']

print("Train:", X_train.shape, y_train.sum(), "fraud")
print("Calibration:", X_calib.shape, y_calib.sum(), "fraud")
print("Test:", X_test.shape, y_test.sum(), "fraud")

Train: (170235, 30) 284 fraud
Calibration: (56745, 30) 94 fraud
Test: (56746, 30) 95 fraud


## 2. Model 1 — Logistic Regression (Interpretable Baseline)

Train Logistic Regression with `class_weight='balanced'` (our chosen imbalance strategy from Layer 1). This is the fast, interpretable floor every other model should beat.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report

logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logreg.fit(X_train, y_train)

logreg_proba = logreg.predict_proba(X_test)[:, 1]
logreg_pr_auc = average_precision_score(y_test, logreg_proba)

print(f"Logistic Regression PR-AUC: {logreg_pr_auc:.4f}")

Logistic Regression PR-AUC: 0.7309


## 3. Model 2 — XGBoost (Primary Model)

Train XGBoost, the primary model for this project. Use `scale_pos_weight` to handle the class imbalance — XGBoost's native equivalent to `class_weight='balanced'`, calculated as the ratio of negative to positive class counts in the training set.

In [3]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

xgb_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    random_state=42
)
xgb_model.fit(X_train, y_train)

xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_pr_auc = average_precision_score(y_test, xgb_proba)

print(f"XGBoost PR-AUC: {xgb_pr_auc:.4f}")

scale_pos_weight: 598.42
XGBoost PR-AUC: 0.8568


## 4. Model 3 — LightGBM (Speed Comparison)

Train LightGBM as a faster alternative to XGBoost, using the same imbalance-handling approach (`scale_pos_weight`), to compare speed vs accuracy trade-offs on this dataset size.

In [4]:
from lightgbm import LGBMClassifier
import time

lgbm_model = LGBMClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbose=-1
)

start = time.time()
lgbm_model.fit(X_train, y_train)
train_time = time.time() - start

lgbm_proba = lgbm_model.predict_proba(X_test)[:, 1]
lgbm_pr_auc = average_precision_score(y_test, lgbm_proba)

print(f"LightGBM PR-AUC: {lgbm_pr_auc:.4f}")
print(f"LightGBM training time: {train_time:.2f}s")

LightGBM PR-AUC: 0.0612
LightGBM training time: 0.82s


## Note: LightGBM Excluded from Comparison

LightGBM produced consistently poor PR-AUC (~0.06) across multiple configurations (default, class-weighted, constrained trees, single-threaded), while identical training data produced strong results with XGBoost (0.8568) and Logistic Regression (0.7309). Diagnosis ruled out overfitting (train-set PR-AUC was equally low) and confirmed a correctly-installed, correctly-referenced package. This is consistent with a known class of LightGBM wheel issues on Apple Silicon (ARM64) macOS, where multi-threaded gradient computation can silently produce incorrect results without raising errors.

**Decision:** LightGBM is excluded from the final model comparison. XGBoost — the project's designated primary model — is unaffected and used going forward.

## 5. Model 4 — Isolation Forest (Unsupervised Anomaly Baseline)

Isolation Forest doesn't use fraud labels during training — it isolates anomalies based on structure alone (anomalies require fewer random splits to isolate than normal points). This tests how well fraud separates from legitimate transactions using no label information at all, as a baseline comparison against the supervised models.

In [5]:
from sklearn.ensemble import IsolationForest

# contamination = expected proportion of anomalies, matches our known fraud rate
contamination_rate = y_train.sum() / len(y_train)
print(f"Contamination rate: {contamination_rate:.5f}")

iso_forest = IsolationForest(contamination=contamination_rate, random_state=42)
iso_forest.fit(X_train)  # note: no y_train — unsupervised

# decision_function: higher = more normal, lower = more anomalous
# we flip the sign so higher score = more like fraud, consistent with other models
iso_scores = -iso_forest.decision_function(X_test)
iso_pr_auc = average_precision_score(y_test, iso_scores)

print(f"Isolation Forest PR-AUC: {iso_pr_auc:.4f}")

Contamination rate: 0.00167
Isolation Forest PR-AUC: 0.1507


## Summary — Model Comparison

| Model | PR-AUC |
|---|---|
| Logistic Regression | 0.7309 |
| **XGBoost** | **0.8568** |
| LightGBM | Excluded (environment issue, documented above) |
| Isolation Forest (unsupervised) | 0.1507 |

**Conclusion:** XGBoost is selected as the primary model, achieving the highest PR-AUC (0.8568) — a ~17% relative improvement over Logistic Regression. Isolation Forest's much lower unsupervised score (0.1507) confirms that label information is essential for this task; fraud is not simply detectable as a structural anomaly. XGBoost will be used going forward for SHAP explainability (Layer 3), conformal prediction (Layer 4), and the FastAPI backend (Layer 6).